# Spindle — Composite Domain

**v1.3.0** introduces `CompositeDomain` for generating data that spans multiple verticals in a single run, with cross-domain foreign keys automatically enforced.

Use this when your Fabric solution has tables from more than one business domain — for example:
- **HR + Retail**: employees who are also loyalty customers
- **Healthcare + Insurance**: patients cross-referenced to policyholders
- **Financial + Retail + HR**: full enterprise data platform

The `SharedEntityRegistry` maps real-world concepts (`PERSON`, `LOCATION`, `ORGANIZATION`, `CALENDAR`) to their domain-specific tables, so Spindle can wire FKs automatically.

In [1]:
%pip install sqllocks-spindle --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\suref\OneDrive\VSCode\AzureClients\forge-workspace\projects\fabric-datagen\.venv-win\Scripts\python.exe -m pip install --upgrade pip


In [2]:
from sqllocks_spindle.domains.retail.retail import RetailDomain
from sqllocks_spindle.domains.hr.hr import HrDomain
from sqllocks_spindle.domains.financial.financial import FinancialDomain
from sqllocks_spindle.domains.healthcare.healthcare import HealthcareDomain
from sqllocks_spindle.domains.composite import CompositeDomain
from sqllocks_spindle.domains.shared_registry import SharedEntityRegistry, SharedConcept
from sqllocks_spindle.engine.generator import Spindle

print('Spindle loaded. Available SharedConcept values:')
for c in SharedConcept:
    print(f'  {c.value}')

Spindle loaded. Available SharedConcept values:
  person
  location
  organization
  calendar


## 1. Retail + HR — Shared PERSON

Generate retail customers and HR employees where the HR employee table is the canonical source of truth for person data. Retail customer records carry a foreign key back to `hr.employee`.

In [3]:
composite_rh = CompositeDomain(
    domains=[RetailDomain(), HrDomain()],
    shared_entities={
        'person': {
            'primary': 'hr.employee',
            'links': {
                'retail': 'customer.employee_id',
            },
        },
    },
)

result_rh = Spindle().generate(domain=composite_rh, scale='fabric_demo', seed=42)
print(result_rh)

GenerationResult(18 tables, 6,545 total rows, 0.2s)


In [4]:
print('Tables generated:')
for name, df in result_rh.tables.items():
    print(f'  {name:<30} {len(df):>6,} rows')

Tables generated:
  hr_department                      30 rows
  hr_position                        80 rows
  hr_employee                       100 rows
  hr_compensation                   300 rows
  hr_performance_review             250 rows
  hr_termination                     15 rows
  hr_time_off_request               500 rows
  hr_training                       100 rows
  hr_training_enrollment            400 rows
  retail_customer                   200 rows
  retail_address                    300 rows
  retail_product_category            50 rows
  retail_product                    100 rows
  retail_promotion                  300 rows
  retail_store                      150 rows
  retail_order                    1,000 rows
  retail_order_line               2,500 rows
  retail_return                     170 rows


In [5]:
# Verify cross-domain FK: retail customer.employee_id -> hr employee.employee_id
tables = result_rh.tables

if 'employee' in tables and 'customer' in tables:
    emp_ids = set(tables['employee']['employee_id'].dropna().unique())
    cust_fk = tables['customer'].get('employee_id')
    if cust_fk is not None:
        linked  = cust_fk.dropna()
        orphans = linked[~linked.isin(emp_ids)]
        print(f'HR employees:             {len(emp_ids):,}')
        print(f'Retail->HR FK references: {len(linked):,}')
        print(f'Orphan references:        {len(orphans):,}  (must be 0)')
    else:
        print('employee_id FK not present in customer table (depends on domain config)')
else:
    print(f'Available tables: {sorted(tables.keys())}')

Available tables: ['hr_compensation', 'hr_department', 'hr_employee', 'hr_performance_review', 'hr_position', 'hr_termination', 'hr_time_off_request', 'hr_training', 'hr_training_enrollment', 'retail_address', 'retail_customer', 'retail_order', 'retail_order_line', 'retail_product', 'retail_product_category', 'retail_promotion', 'retail_return', 'retail_store']


## 2. Retail + HR + Financial — Three-domain composite

Add a third domain. Shared PERSON (HR → Retail + Financial) and shared LOCATION (Retail store → Financial branch).

In [6]:
composite_3 = CompositeDomain(
    domains=[RetailDomain(), HrDomain(), FinancialDomain()],
    shared_entities={
        'person': {
            'primary': 'hr.employee',
            'links': {
                'retail':    'customer.employee_id',
                'financial': 'customer.employee_id',
            },
        },
        'location': {
            'primary': 'retail.store',
            'links': {
                'financial': 'branch.store_id',
            },
        },
    },
)

result_3 = Spindle().generate(domain=composite_3, scale='fabric_demo', seed=42)
print(f'Tables: {len(result_3.tables)}')
print(f'Total rows: {sum(len(df) for df in result_3.tables.values()):,}')
print(f'\nChild domains: {[d.__class__.__name__ for d in composite_3.child_domains]}')

Tables: 28
Total rows: 10,141

Child domains: ['RetailDomain', 'HrDomain', 'FinancialDomain']


In [7]:
import pandas as pd

summary = pd.DataFrame([
    {'table': k, 'rows': len(v), 'columns': len(v.columns)}
    for k, v in result_3.tables.items()
]).sort_values('rows', ascending=False)
summary

,table,rows,columns
26,retail_order_line,2500,8
22,financial_statement,1320,8
25,retail_order,1000,8
23,financial_transaction,1000,9
7,hr_time_off_request,500,7
21,financial_loan_payment,480,7
9,hr_training_enrollment,400,7
14,retail_promotion,300,6
4,hr_compensation,300,6
11,retail_address,300,10


## 3. Auto-registry

When you don't provide `shared_entities`, `SharedEntityRegistry` applies built-in default concept mappings automatically. Great for quick multi-domain exploration.

In [8]:
registry = SharedEntityRegistry()

composite_auto = CompositeDomain(
    domains=[RetailDomain(), HealthcareDomain()],
    registry=registry,
)

result_auto = Spindle().generate(domain=composite_auto, scale='fabric_demo', seed=42)
print(result_auto)
print('\nTables:', sorted(result_auto.tables.keys()))

GenerationResult(18 tables, 8,952 total rows, 0.1s)

Tables: ['healthcare_claim', 'healthcare_claim_line', 'healthcare_diagnosis', 'healthcare_encounter', 'healthcare_facility', 'healthcare_medication', 'healthcare_patient', 'healthcare_procedure', 'healthcare_provider', 'retail_address', 'retail_customer', 'retail_order', 'retail_order_line', 'retail_product', 'retail_product_category', 'retail_promotion', 'retail_return', 'retail_store']


## 4. Preview composite data (local) / Write to Fabric Delta Tables

In a **Fabric Spark notebook**, use `spark.createDataFrame(df)` and `.saveAsTable()` to write composite tables to the Lakehouse. Below we show a local pandas preview instead.

In [9]:
# Local preview — in Fabric, replace this with spark.createDataFrame(df).write...saveAsTable()
for table_name, df in result_3.tables.items():
    print(f'  composite_{table_name:<30} {len(df):>6,} rows  {len(df.columns):>2} cols')

print(f'\nAll {len(result_3.tables)} composite domain tables ready.')
print('In Fabric: use spark.createDataFrame(df).write.format("delta").saveAsTable(...)')

  composite_financial_transaction_category     40 rows   4 cols
  composite_hr_department                      30 rows   4 cols
  composite_hr_position                        80 rows   6 cols
  composite_hr_employee                       100 rows  11 cols
  composite_hr_compensation                   300 rows   6 cols
  composite_hr_performance_review             250 rows   7 cols
  composite_hr_termination                     15 rows   6 cols
  composite_hr_time_off_request               500 rows   7 cols
  composite_hr_training                       100 rows   5 cols
  composite_hr_training_enrollment            400 rows   7 cols
  composite_retail_customer                   200 rows   9 cols
  composite_retail_address                    300 rows  10 cols
  composite_retail_product_category            50 rows   4 cols
  composite_retail_product                    100 rows   6 cols
  composite_retail_promotion                  300 rows   6 cols
  composite_retail_store                

In [10]:
# Local pandas equivalent of cross-domain Spark SQL query:
# "employees who are also retail customers"
# In Fabric, you would use spark.sql(...) against the Delta tables instead.

tables = result_3.tables
emp = tables['hr_employee']
cust = tables['retail_customer']
orders = tables['retail_order']

# Show sample data from each cross-domain table
print('HR Employees (first 3):')
print(emp[['employee_id', 'first_name', 'last_name']].head(3).to_string(index=False))
print(f'\nRetail Customers: {len(cust):,} rows')
print(f'Retail Orders:    {len(orders):,} rows')
print(f'\nIn Fabric, run cross-domain Spark SQL to join these tables via shared FKs.')

HR Employees (first 3):
 employee_id first_name last_name
           1      James    Kramer
           2     Rachel     Blair
           3      Craig    Turner

Retail Customers: 200 rows
Retail Orders:    1,000 rows

In Fabric, run cross-domain Spark SQL to join these tables via shared FKs.
